# 1. Escolha do Município e Obtenção dos Dados:

In [ ]:
import folium
import pandas as pd
import numpy as np

from folium.plugins import Fullscreen, MiniMap
from branca.colormap import LinearColormap

# Leitura dos DataFrames de Ouro Branco (Votação das Sessões Eleitorais e Local de Votação dos Eleitorados):
df_voting_session = pd.read_csv('votacao_secao_2026_MG_Ouro_Branco.csv', sep=';', encoding='latin_1')
df_electorate_voting_location = pd.read_csv('eleitorado_local_votacao_2026_MG_Ouro_Branco.csv', sep=';', encoding='latin_1')

display(df_voting_session.head())

# 2. Limpeza, Filtragem e Unificação de Dados:

In [ ]:
# Adquirindo a localização das sessões eleitorais:

# Padronizando as chaves geográficas, selecionando as colunas para obter a localização de cada seção nas zonas eleitorais:
df_location = df_electorate_voting_location[
    ['NR_TURNO', 'NR_ZONA', 'NR_SECAO', 'NR_LATITUDE', 'NR_LONGITUDE']
    ].drop_duplicates(subset=['NR_TURNO', 'NR_ZONA', 'NR_SECAO'])

# Convertendo as coordenadas para valores numéricos:
df_location['NR_LATITUDE'] = df_location['NR_LATITUDE'].astype(str).str.replace(',', '.')
df_location['NR_LONGITUDE'] = df_location['NR_LONGITUDE'].astype(str).str.replace(',', '.')

df_location['NR_LATITUDE'] = pd.to_numeric(df_location['NR_LATITUDE'], errors='coerce')
df_location['NR_LONGITUDE'] = pd.to_numeric(df_location['NR_LONGITUDE'], errors='coerce')

# Removendo coordenadas nulas ou com o valor -1:
df_location = df_location.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE'])
df_location = df_location[(df_location['NR_LATITUDE'] != -1) & (df_location['NR_LONGITUDE'] != -1)]

# DataFrame com todos os dados que serão utilizados:
df = df_voting_session[['NR_TURNO', 'CD_ELEICAO', 'NR_ZONA', 'NR_SECAO', 'CD_CARGO', 
                        'DS_CARGO', 'NR_VOTAVEL', 'NM_VOTAVEL', 'QT_VOTOS', 'NM_LOCAL_VOTACAO', 'DS_LOCAL_VOTACAO_ENDERECO']].copy()

# Fazendo o merge das colunas 'NR_LATITUDE' e 'NR_LONGITUDE' no DataFrame:
df = df.merge(df_location, on=['NR_TURNO', 'NR_ZONA', 'NR_SECAO'], how='left')

# Agrupando a contagem total de votos por candidato em cada seção eleitoral nos dois turnos:
total_vote_count_grouped = df.groupby(
    ['NR_TURNO', 'NR_ZONA', 'NR_SECAO', 'NR_VOTAVEL', 'NM_VOTAVEL', 'NM_LOCAL_VOTACAO', 
     'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE'],
     as_index=False
    )['QT_VOTOS'].sum()

# 3. Mapeamento Graduado do 1º Turno:

In [ ]:
# Filtrando os dados do primeiro turno:
df_1t = total_vote_count_grouped[
    total_vote_count_grouped['NR_TURNO'].astype(str) == '1'
].copy()

# Removendo votos brancos e nulos:
df_1t = df_1t[
    ~df_1t['NM_VOTAVEL'].str.contains('BRANCO|NULO', case=False)
].copy()

total_vote_candidates = (
    df_1t.groupby('NM_VOTAVEL')['QT_VOTOS']
    .sum()
    .sort_values(ascending=False)
)

candidates = total_vote_candidates.head(5).index.tolist()

display(total_vote_candidates.head(5))

# Organizando os votos por seção eleitoral e por candidato:
votes_1t = df_1t.pivot_table(
        index=['NR_ZONA', 'NR_SECAO'],
        columns='NM_VOTAVEL',
        values='QT_VOTOS',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

votes_1t.columns.name = None

# Adicionando as informações de localização de cada seção:
df_info_location = df_1t[
    ['NR_ZONA', 'NR_SECAO', 'NM_LOCAL_VOTACAO', 
     'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LATITUDE', 'NR_LONGITUDE']
     ].drop_duplicates(subset=['NR_ZONA', 'NR_SECAO'])

votes_1t = votes_1t.merge(df_info_location, on=['NR_ZONA', 'NR_SECAO'], how='left', validate='one_to_one')

# Calculando o total de votos válidos por seção:
candidates_columns = [c for c in votes_1t.columns if c in df_1t['NM_VOTAVEL'].unique()]

votes_1t['TOTAL_VALIDOS'] = votes_1t[candidates_columns].sum(axis=1)

# Calculando o percentual de votos de cada candidato por seção:
for candidate in candidates:
    votes_1t[f'PERCENTUAL_{candidate}'] = np.where(votes_1t['TOTAL_VALIDOS'] > 0, votes_1t[candidate] / votes_1t['TOTAL_VALIDOS'] * 100, 0)

display(votes_1t.head())

palettes = [
    '#272D97',
    '#D60828',
    '#4C8790',
    '#D8942F',
    '#A0A00B'
]

# Associando uma cor aos candidatos:
candidates_palettes = {
    candidate: palettes[i % len(palettes)]
    for i, candidate in enumerate(candidates)
}

def create_map_by_candidate(df, candidate, palette):
    """Função utilizada para criar um mapa para cada candidato, utilizando a latitude e longitude das seções eleitorais."""

    df_map = df.dropna(subset=['NR_LATITUDE', 'NR_LONGITUDE']).copy()
    df_map = df_map[(df_map['NR_LATITUDE'] != -1) & (df_map['NR_LONGITUDE'] != -1)]

    center = [-20.5233, -43.6914]

    map = folium.Map(location=center, zoom_start=12, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors')

    scale = LinearColormap(colors=['#ffffff', palette], vmin=0, vmax=100)
    scale.caption = f'Percentual de votos - {candidate} (%)'

    for _, row in df_map.iterrows():

        percentage = row[f'PERCENTUAL_{candidate}']
        vote_number = int(row[candidate])

        html_content = f"""
            <div style="
                width: 250px;
                font-family: Arial, sans-serif;
                padding: 5px;
            ">
                <h4 style="
                    color: {palette};
                    margin-bottom: 10px;
                    border-bottom: 2px solid {palette};
                    padding-bottom: 6px;
                ">
                    {candidate}
                </h4>

                <p><b>Local de votação:</b><br>
                {row['NM_LOCAL_VOTACAO']}</p>

                <p><b>Endereço:</b><br>
                {row['DS_LOCAL_VOTACAO_ENDERECO']}</p>

                <p>
                    <b>Zona eleitoral:</b> {row['NR_ZONA']}<br>
                    <b>Seção:</b> {row['NR_SECAO']}
                </p>

                <div style="
                    background-color: {palette};
                    color: white;
                    padding: 10px;
                    text-align: center;
                    border-radius: 6px;
                    margin-top: 10px;
                ">
                    <span style="font-size: 12px;">
                        Percentual de votos
                    </span><br>
                    <b style="font-size: 22px;">
                        {percentage:.2f}%
                    </b>
                </div>
            </div>
        """

        popup = folium.Popup(html_content, max_width=300)

        folium.CircleMarker(
            location=[row['NR_LATITUDE'], row['NR_LONGITUDE']],
            radius=8,
            color=palette,
            weight=1,
            fill=True,
            fill_color=palette,
            fill_opacity=0.85,
            popup=popup,
            tooltip=f'{candidate}: {percentage:.2f}%'
        ).add_to(map)
    scale.add_to(map)

    MiniMap(toggle_display=True).add_to(map)

    return map

maps_1t = {}

for candidate in candidates:
    palette = candidates_palettes[candidate]

    map = create_map_by_candidate(votes_1t, candidate, palette)
    maps_1t[candidate] = map

    filename = candidate.lower().replace(" ", "_")
    output_path = f"mapa_1t_{filename}.html"

    map.save(output_path)

# 4. Análise do 2º Turno e Comparação Espacial:

In [ ]:
# 1. Definindo o servidor de terreno (OpenTopoMap)
tiles_terreno = 'https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png'
attr_terreno = 'Map data: &copy; OpenStreetMap contributors, SRTM | Map style: &copy; OpenTopoMap'

# 2. Criando o mapa centralizado em uma coordenada (Ex: Rio de Janeiro)
mapa = folium.Map(
    location=[-22.9068, -43.1729],
    zoom_start=10,
    tiles=tiles_terreno,
    attr=attr_terreno
)

# 3. visualizando o mapa
mapa

In [ ]:
# Criar mapa base
mapa_marcadores = folium.Map(location=[-15.793889, -47.882778], zoom_start=12, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', # Servidor alternativo
    attr='&copy; OpenStreetMap contributors')

# Marcador 1: Esplanada dos Ministérios
folium.Marker(
    location=[-15.7981, -47.8756],
    popup="<b>Esplanada dos Ministérios</b><br>Centro do Governo Federal",
    tooltip="Clique para saber mais",
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(mapa_marcadores)

# Marcador 2: Congresso Nacional (Ícone personalizado)
folium.Marker(
    location=[-15.7998, -47.8642],
    popup="<b>Congresso Nacional</b>",
    tooltip="Congresso Nacional",
    icon=folium.Icon(color='green', icon='star')
).add_to(mapa_marcadores)

mapa_marcadores

In [ ]:
mapa_circulos = folium.Map(location=[-15.793889, -47.882778], zoom_start=13, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', # Servidor alternativo
    attr='&copy; OpenStreetMap contributors')

# CircleMarker: Raio fixo em Pixels (útil para pontos agregados)
folium.CircleMarker(
    location=[-15.7981, -47.8756],
    radius=15,
    popup="Área Central (Raio Fixo)",
    color='blue',
    fill=True,
    fill_color='lightblue',
    fill_opacity=0.6
).add_to(mapa_circulos)

# Circle: Raio real em Metros (útil para raio de cobertura)
folium.Circle(
    location=[-15.793889, -47.882778],
    radius=2000, # 2 km de raio
    popup="Zona de Cobertura de 2km",
    color='red',
    fill=True,
    fill_opacity=0.2
).add_to(mapa_circulos)

mapa_circulos

In [ ]:
from folium import plugins
mapa_plugins = folium.Map(location=[-15.793889, -47.882778], zoom_start=11, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', # Servidor alternativo
    attr='&copy; OpenStreetMap contributors')

# Dados simulados de ocorrências/pontos (Latitude, Longitude)
dados_pontos = [
    [-15.7981, -47.8756], [-15.7998, -47.8642], [-15.7800, -47.8900],
    [-15.7850, -47.8850], [-15.8000, -47.8700], [-15.8100, -47.8600],
    [-15.8200, -47.8950], [-15.7700, -47.8800], [-15.7750, -47.8700]
]

# 1. Agrupamento de Marcadores (MarkerCluster)
cluster = plugins.MarkerCluster(name="Agrupamento de Pontos").add_to(mapa_plugins)
for lat_lng in dados_pontos:
    folium.Marker(location=lat_lng).add_to(cluster)

# 2. Mapa de Calor (HeatMap)
plugins.HeatMap(dados_pontos, name="Mapa de Calor").add_to(mapa_plugins)

# Adicionar Controle de Camadas para ligar/desligar
folium.LayerControl().add_to(mapa_plugins)

mapa_plugins

In [ ]:
# 1. Criar mapa principal
mapa_utilitarios = folium.Map(location=[-15.793889, -47.882778], zoom_start=12, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', # Servidor alternativo
    attr='&copy; OpenStreetMap contributors')


# 2. Criar a camada de tiles pública explicitamente
camada_mini = folium.TileLayer(
    tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors'
)

# 3. Passar a camada para o MiniMap
minimap = plugins.MiniMap(
    tile_layer=camada_mini,
    toggle_display=True,
    position='bottomright'
)

# 4. Adicionar ao mapa principal
mapa_utilitarios.add_child(minimap)

# Régua de Medição de Distância/Área
mapa_utilitarios.add_child(plugins.MeasureControl(position='topright'))

# Popup para mostrar coordenadas ao clicar
mapa_utilitarios.add_child(folium.LatLngPopup())

mapa_utilitarios

In [ ]:
# 1. Carregar Dados de Desemprego nos EUA
url = 'https://raw.githubusercontent.com/python-visualization/folium/main/examples/data'
df_desemprego = pd.read_csv(f'{url}/US_Unemployment_Oct2012.csv')
geo_json_estados = f'{url}/us-states.json'

print("Primeiras linhas do DataFrame:")
print(df_desemprego.head())

# 2. Criar Mapa Base dos EUA
mapa_choropleth = folium.Map(location=[48, -102], zoom_start=4, tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png',
    attr='&copy; OpenStreetMap contributors' )

# 3. Adicionar a Camada Choropleth
folium.Choropleth(
    geo_data=geo_json_estados,
    name='Taxa de Desemprego',
    data=df_desemprego,
    columns=['State', 'Unemployment'],
    key_on='feature.id',            # Chave de ligação no GeoJSON
    fill_color='YlOrRd',            # Escala Amarelo-Laranja-Vermelho
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Taxa de Desemprego (%)'
).add_to(mapa_choropleth)

folium.LayerControl().add_to(mapa_choropleth)

mapa_choropleth

In [ ]:
# Salvar o mapa final como um arquivo HTML estático/interativo
mapa_choropleth.save("mapa_desemprego_eua.html")
print("Mapa salvo com sucesso em 'mapa_desemprego_eua.html'!")

In [ ]:
import folium
import pandas as pd

# 1. Dados de exemplo (Estados e um valor numérico fictício, ex: Pontuação/Índice)
dados_estados = pd.DataFrame({
    'UF': ['SP', 'RJ', 'MG', 'BA', 'RS', 'PR', 'PE', 'CE', 'PA', 'AM'],
    'Valor': [95.5, 82.3, 88.0, 75.1, 84.6, 86.2, 72.4, 70.8, 65.0, 68.3]
})

# 2. URL do GeoJSON com a malha dos Estados do Brasil
url_geojson = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'

# 3. Criar o Mapa Base centralizado no Brasil
mapa_br = folium.Map(location=[-14.2350, -51.9253], zoom_start=4,tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', attr='&copy; OpenStreetMap contributors')

# 4. Adicionar a camada Choropleth
folium.Choropleth(
    geo_data=url_geojson,
    name='Choropleth Brasil',
    data=dados_estados,
    columns=['UF', 'Valor'],
    key_on='feature.properties.sigla',  # A propriedade 'sigla' no GeoJSON contém 'SP', 'RJ', etc.
    fill_color='YlGnBu',                 # Escala de cores (Amarelo, Verde, Azul)
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Índice por Estado'
).add_to(mapa_br)

# Adicionar controle para ligar/desligar camadas
folium.LayerControl().add_to(mapa_br)

mapa_br

In [ ]:
import folium
import pandas as pd

# 1. Dados dos Estados (Exemplo)
dados_estados = pd.DataFrame({
    'UF': ['SP', 'RJ', 'MG', 'BA', 'RS', 'PR', 'PE', 'CE', 'PA', 'AM'],
    'Nome_Estado': ['São Paulo', 'Rio de Janeiro', 'Minas Gerais', 'Bahia',
                   'Rio Grande do Sul', 'Paraná', 'Pernambuco', 'Ceará', 'Pará', 'Amazonas'],
    'Valor': [95.5, 82.3, 88.0, 75.1, 84.6, 86.2, 72.4, 70.8, 65.0, 68.3]
})

# 2. GeoJSON com a malha dos Estados do Brasil
url_geojson = 'https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson'

# 3. Criar o Mapa Base
mapa = folium.Map(location=[-14.2350, -51.9253], zoom_start=4,tiles='https://tile.openstreetmap.de/{z}/{x}/{y}.png', attr='&copy; OpenStreetMap contributors')

# 4. Criar a camada Choropleth para a coloração graduada
choropleth = folium.Choropleth(
    geo_data=url_geojson,
    name='Choropleth Brasil',
    data=dados_estados,
    columns=['UF', 'Valor'],
    key_on='feature.properties.sigla',
    fill_color='YlGnBu',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Índice por Estado'
).add_to(mapa)

estilo_tooltip = """
    background-color: #2b2b2b;
    color: #ffffff;
    font-family: 'Helvetica Neue', Arial, sans-serif;
    font-size: 13px;
    border-radius: 6px;
    border: 1px solid #444444;
    padding: 8px 12px;
    box-shadow: 3px 3px 10px rgba(0, 0, 0, 0.4);
"""

# Criando o GeoJsonTooltip estilizado
tooltip_customizado = folium.GeoJsonTooltip(
    fields=['name', 'sigla'],             # Propriedades presentes no arquivo GeoJSON
    aliases=['Estado:', 'UF:'],            # Rótulos amigáveis exibidos no balão
    style=estilo_tooltip,                   # CSS aplicado
    sticky=True,                            # Acompanha o ponteiro do mouse
    localize=True                           # Formatação local de números
)
folium.GeoJson(url_geojson, tooltip=tooltip_customizado).add_to(mapa)
mapa

In [ ]:
import folium

# 1. Inicializa o mapa sem nenhum tile padrão
mapa = folium.Map(
    location=[-23.55052, -46.633308],  # São Paulo
    zoom_start=12,
    tiles=None
)

# 2. Define e adiciona as 3 opções de tiles XYZ

# Opção 1: CyclOSM (Estilo ciclístico/topográfico)
url_cyclosm = 'https://{s}.tile-cyclosm.openstreetmap.fr/cyclosm/{z}/{x}/{y}.png'
attr_cyclosm = '&copy; CyclOSM via OpenStreetMap'
folium.TileLayer(
    tiles=url_cyclosm,
    attr=attr_cyclosm,
    name="Ciclovias e Relevo (CyclOSM)",
    control=True
).add_to(mapa)

# Opção 2: Humanitarian OpenStreetMap (Estilo infraestrutura/colorido)
url_humanitarian = 'https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png'
attr_humanitarian = '&copy; OpenStreetMap contributors, Humanitarian OpenStreetMap Team'
folium.TileLayer(
    tiles=url_humanitarian,
    attr=attr_humanitarian,
    name="Humanitário (OSM HOT)",
    control=True
).add_to(mapa)


# 3. Adiciona o controle de camadas (menu interativo)

folium.LayerControl(position="topright", collapsed=False).add_to(mapa)

mapa